In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_date, lit

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b56d52b2-34d3-4758-b8e8-1381ddd28b1c;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 363ms :: artifacts dl 18ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:
tb_orders_and_weather = spark.read.csv("s3a://last-mile-optimization-trusted/dataset-join/join_orders_and_weather.csv/", header=True, inferSchema=True)
tb_dates = spark.read.csv("s3a://last-mile-optimization-trusted/dataset-holidays/dates_2016_2018/", header=True, inferSchema=True)

26/04/19 03:27:10 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
from pyspark.sql import functions as F

# 1. Preparar tb_dates (ajustando o formato brasileiro)
tb_dates_prep = tb_dates.withColumn("data_feriado_formatada", F.to_date(F.col("date"), "dd/MM/yyyy"))

# 2. Preparar tb_orders_and_weather (extraindo apenas a data para o join)
tb_orders_prep = tb_orders_and_weather.withColumn(
    "shipping_date_only", 
    F.to_date(F.col("order_delivered_carrier_date"))
)

# 3. Join e criação da coluna holiday_at_shipping
df_final = tb_orders_prep.join(
    tb_dates_prep, 
    tb_orders_prep.shipping_date_only == tb_dates_prep.data_feriado_formatada, 
    "left"
).withColumn(
    "holiday_at_shipping", 
    F.when(F.col("data_feriado_formatada").isNotNull(), 1).otherwise(0)
)

# 4. Seleção final com a ordem desejada e aliases (os drops ocorrem aqui por omissão)
df_final = df_final.select(
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "holiday_at_shipping", # Mantida do seu código original
    "approval_time_hours",
    "handling_time_hours",
    "shipping_delay_hours",
    "delivery_time_days",
    "purchase_hour",
    "purchase_day_of_week",
    "is_weekend",
    "price",
    "freight_value",
    "product_weight_g",
    "volume_cm3",
    "distance_km",
    "same_city",
    "review_score",
    "delivered_on_time",
    "seller_geolocation_lat",
    "seller_geolocation_lng",
    "seller_total_rainfall_period_mm",
    "seller_max_rain_intensity_mm_h",
    "seller_max_wind_gust_period_ms",
    "seller_interval_code",
    "seller_rain_class_code",
    "seller_wind_class_code",
    "customer_geolocation_lat",
    "customer_geolocation_lng",
    "customer_total_rainfall_period_mm",
    "customer_max_rain_intensity_mm_h",
    "customer_max_wind_gust_period_ms",
    "customer_interval_code",
    "customer_rain_class_code",
    "customer_wind_class_code",
    "month",
    "seller_accumulated_rainfall_3_days_mm",
    "seller_is_heavy_rain",
    "seller_is_strong_wind",
    "customer_accumulated_rainfall_3_days_mm",
    "customer_is_heavy_rain",
    "customer_is_strong_wind",
    "seller_meso_metropolitana_de_sao_paulo",
    "seller_meso_araraquara",
    "seller_meso_piracicaba",
    "seller_meso_ribeirao_preto",
    "seller_meso_sao_jose_do_rio_preto",
    "seller_meso_campinas",
    "seller_meso_vale_do_paraiba_paulista",
    "seller_meso_macro_metropolitana_paulista",
    "seller_meso_bauru",
    "seller_meso_presidente_prudente",
    "seller_meso_marilia",
    "seller_meso_assis",
    "seller_meso_aracatuba",
    "seller_meso_litoral_sul_paulista",
    "seller_meso_itapetininga",
    "customer_meso_metropolitana_de_sao_paulo",
    "customer_meso_piracicaba",
    "customer_meso_campinas",
    "customer_meso_vale_do_paraiba_paulista",
    "customer_meso_ribeirao_preto",
    "customer_meso_sao_jose_do_rio_preto",
    "customer_meso_macro_metropolitana_paulista",
    "customer_meso_bauru",
    "customer_meso_araraquara",
    "customer_meso_presidente_prudente",
    "customer_meso_marilia",
    "customer_meso_assis",
    "customer_meso_aracatuba",
    "customer_meso_itapetininga",
    "customer_meso_litoral_sul_paulista"
)

# Exibir para conferência
df_final.show(5)

26/04/19 03:27:28 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------------------------+-----------------------------+-------------------+-------------------+-------------------+--------------------+------------------+-------------+--------------------+----------+-----+-------------+----------------+----------+-----------+---------+------------+-----------------+----------------------+----------------------+-------------------------------+------------------------------+------------------------------+--------------------+----------------------+----------------------+------------------------+------------------------+---------------------------------+--------------------------------+--------------------------------+----------------------+------------------------+------------------------+-----+-------------------------------------+--------------------+---------------------+---------------------------------------+----------------------+-----------------------+--------------------------------------+----------------------+----------------------+----

In [5]:
df_final.groupBy("holiday_at_shipping").count().show()

+-------------------+-----+
|holiday_at_shipping|count|
+-------------------+-----+
|                  1|   29|
|                  0|33791|
+-------------------+-----+



In [6]:
# Filtra apenas onde é feriado e mostra as primeiras 20 linhas
df_final.filter(F.col("holiday_at_shipping") == 1).show()

+----------------------------+-----------------------------+-------------------+-------------------+-------------------+--------------------+------------------+-------------+--------------------+----------+------+-------------+----------------+----------+-----------+---------+------------+-----------------+----------------------+----------------------+-------------------------------+------------------------------+------------------------------+--------------------+----------------------+----------------------+------------------------+------------------------+---------------------------------+--------------------------------+--------------------------------+----------------------+------------------------+------------------------+-----+-------------------------------------+--------------------+---------------------+---------------------------------------+----------------------+-----------------------+--------------------------------------+----------------------+----------------------+---

In [7]:
df_final.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-client/table_orders_weather_holidays.csv')

spark.stop()

26/04/19 03:27:40 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/19 03:27:40 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
